# 🦆 DuckDB ETL Demo
**YZV 322E — Applied Data Engineering | Spring 2026**

This notebook walks through all four stages of the demo interactively.

In [ ]:
import duckdb
import pandas as pd
import os

BASE_DIR = os.path.abspath('..')
CSV_PATH = os.path.join(BASE_DIR, 'data', 'sales.csv')
PARQUET_PATH = os.path.join(BASE_DIR, 'data', 'sales.parquet')

print('DuckDB version:', duckdb.__version__)

## 1. Basic Queries — SQL directly on CSV

In [ ]:
duckdb.sql(f"""
    SELECT region, ROUND(SUM(amount),2) AS revenue, COUNT(*) AS orders
    FROM '{CSV_PATH}'
    WHERE amount IS NOT NULL
    GROUP BY region
    ORDER BY revenue DESC
""").df()

## 2. Pandas Integration — zero-copy

In [ ]:
df = pd.read_csv(CSV_PATH)
print(f'DataFrame shape: {df.shape}')

# Query df directly — no copy!
duckdb.sql("""
    SELECT category, ROUND(AVG(amount),2) AS avg_value
    FROM df
    WHERE amount IS NOT NULL
    GROUP BY category
    ORDER BY avg_value DESC
""").df()

## 3. Parquet — speed & size

In [ ]:
import time

for label, path in [('CSV', CSV_PATH), ('Parquet', PARQUET_PATH)]:
    t0 = time.perf_counter()
    duckdb.sql(f"SELECT SUM(amount) FROM '{path}'").fetchone()
    print(f'{label}: {time.perf_counter()-t0:.4f}s  |  {os.path.getsize(path)/1e6:.1f} MB')

## 4. Full ETL Pipeline

In [ ]:
con = duckdb.connect()

con.execute(f"CREATE TABLE raw AS SELECT * FROM '{CSV_PATH}' WHERE amount IS NOT NULL")

result = con.sql("""
    SELECT region, category,
        ROUND(SUM(amount),2) AS revenue,
        COUNT(*) AS orders
    FROM raw
    GROUP BY region, category
    ORDER BY revenue DESC
    LIMIT 10
""").df()

result